# Realistic Synthetic VI Science Cycle

This notebook visualizes a full synthetic-to-inference science cycle output produced by `scripts/run_realistic_synthetic_vi_cycle.py`.

Run once in terminal first:

```bash
python scripts/run_realistic_synthetic_vi_cycle.py --output-dir outputs/realistic_synthetic_vi_cycle
```


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True


In [ ]:
output_dir = Path('outputs/realistic_synthetic_vi_cycle')
summary = json.loads((output_dir / 'summary.json').read_text())
arr = np.load(output_dir / 'science_cycle_outputs.npz')

print('Sampler:', summary['sampler'])
print('VI converged:', summary['vi']['converged'])
print('Final objective:', summary['vi']['final_objective'])
print('Gradient rel-L2 error:', summary['gradient_check']['relative_l2_error'])


## 1. IC and Ground Truth Diagnostics


In [ ]:
coords = arr['coords_xy']
spaxels = arr['spaxels']
true_age = arr['true_age']
true_met = arr['true_metallicity']

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
sc0 = axs[0].scatter(coords[:, 0], coords[:, 1], c=true_age, s=20)
axs[0].set_title('Particle ICs colored by true age')
plt.colorbar(sc0, ax=axs[0], fraction=0.046)

sc1 = axs[1].scatter(coords[:, 0], coords[:, 1], c=true_met, s=20)
axs[1].set_title('Particle ICs colored by true metallicity')
plt.colorbar(sc1, ax=axs[1], fraction=0.046)

axs[2].hist2d(spaxels[:, 0], spaxels[:, 1], bins=24)
axs[2].set_title('Particle occupancy in spaxel grid')
axs[2].set_xlabel('ix')
axs[2].set_ylabel('iy')

plt.tight_layout()
plt.show()


## 2. Cube-Level Fit Diagnostics


In [ ]:
target = arr['target_cube']
pred = arr['pred_mean_cube']
resid = arr['residual_cube']
chi2 = arr['chi2_cube']
wave_idx = target.shape[2] // 2

fig, axs = plt.subplots(1, 4, figsize=(18, 4))
im0 = axs[0].imshow(target[:, :, wave_idx])
axs[0].set_title(f'Ground truth slice (w={wave_idx})')
plt.colorbar(im0, ax=axs[0], fraction=0.046)

im1 = axs[1].imshow(pred[:, :, wave_idx])
axs[1].set_title('Posterior mean slice')
plt.colorbar(im1, ax=axs[1], fraction=0.046)

im2 = axs[2].imshow(resid[:, :, wave_idx])
axs[2].set_title('Residual slice')
plt.colorbar(im2, ax=axs[2], fraction=0.046)

im3 = axs[3].imshow(chi2[:, :, wave_idx])
axs[3].set_title('Chi2 slice')
plt.colorbar(im3, ax=axs[3], fraction=0.046)

plt.tight_layout()
plt.show()


## 3. Variational Optimization Diagnostics


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4))
axs[0].plot(arr['vi_objective'])
axs[0].set_title('VI objective')

axs[1].plot(arr['vi_reconstruction'])
axs[1].set_title('Reconstruction')

axs[2].plot(arr['vi_kl'])
axs[2].set_title('KL')

axs[3].plot(arr['vi_grad_norm'])
axs[3].set_yscale('log')
axs[3].set_title('Grad norm')

for ax in axs:
    ax.set_xlabel('step')
plt.tight_layout()
plt.show()


## 4. Truth-Recovery Checks


In [ ]:
fit_age = arr['fit_age']
fit_met = arr['fit_metallicity']

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].scatter(true_age, fit_age, s=16)
lims = [min(true_age.min(), fit_age.min()), max(true_age.max(), fit_age.max())]
axs[0].plot(lims, lims, 'k--')
axs[0].set_title('Age recovery')
axs[0].set_xlabel('truth')
axs[0].set_ylabel('fit')

axs[1].scatter(true_met, fit_met, s=16)
lims = [min(true_met.min(), fit_met.min()), max(true_met.max(), fit_met.max())]
axs[1].plot(lims, lims, 'k--')
axs[1].set_title('Metallicity recovery')
axs[1].set_xlabel('truth')
axs[1].set_ylabel('fit')

plt.tight_layout()
plt.show()

print('Recovery metrics:')
for k, v in summary['recovery'].items():
    print(f'  {k}: {v:.6g}')
